# interp-engine adapter: capture and steer gpt2 on CPU

This notebook loads the adapter **from the Hub** rather than importing an installed package, because
that is how the component is delivered and the only path a consumer actually takes. It then does the
two things the adapter exists for: capture activations at named hooks, and apply an intervention at
each of the two position scopes it declares.

Everything here runs on CPU with gpt2. No GPU, no lease, no credentials beyond Hub read access.


## 1. Load the component from the Hub

Two steps, and the separation is deliberate. `pull` fetches and caches **without executing anything**,
so you can read the entrypoint before trusting it. `load_hub_adapter` is what executes, and it refuses
unless you opt in through `IT_TRUST_REMOTE_CODE` — loading an adapter runs published code in your
process and composes it into your session's MRO.


In [ ]:
import os

os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in explicitly; the loader refuses without it

import interpretune as it
from interpretune.hub.adapters import load_hub_adapter, loaded_adapter_module

REPO = "speediedan/it-interp-engine-adapter"

manifest, revision = it.hub.pull(REPO)   # caches; executes nothing
print("revision:", revision[:12])
print("declares:", manifest["adapters"]["declares"])


### What the manifest promised, and what this environment can give you

A component may declare compositions this environment cannot satisfy. Those are **skipped and
reported** rather than silently dropped, so "unavailable here" never looks like "does not exist".
Without `circuit-tracer` installed you will see the conditional compositions reported as unavailable.


In [ ]:
members = load_hub_adapter(REPO)   # executes the entrypoint, behind the trust gate
print("registered adapters:", [getattr(m, "name", str(m)) for m in members])

from interpretune.adapters import ADAPTER_REGISTRY
keys = sorted(
    "+".join(str(getattr(a, "name", a)) for a in k)
    for k in ADAPTER_REGISTRY.available_compositions()
    if any("interp_engine" in str(getattr(a, "name", a)) for a in k)
)
print(f"\nregistered {len(keys)} of {len(manifest['adapters']['compositions'])} declared compositions:")
for k in keys:
    print("   ", k)


## 2. Wrap gpt2 and capture

The adapter reaches every engine symbol through one seam, and hook names are translated there rather
than guessed. Names that would be ambiguous are **refused** with the name to ask for instead — see the
last section.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

seam = loaded_adapter_module(REPO)._Seam
cfg = loaded_adapter_module(REPO).InterpEngineConfig()

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
tok.padding_side = "left"
hf = AutoModelForCausalLM.from_pretrained("gpt2").eval()
engine = seam.wrap_model(hf, tok, cfg, "gpt2")   # wraps IN PLACE; no reload, no second copy

prompt = "The capital of France is"
ids = tok(prompt, return_tensors="pt")["input_ids"]
point = seam.point_for("blocks.5.hook_resid_post", engine)
cache = seam.capture(engine, ids, [point])
print("captured", tuple(cache[point].shape), "at", point)


### The forward agrees with HuggingFace exactly

Not within a tolerance — at `0.0`. The engine wraps the HuggingFace model in place and runs its
forward, so any difference would be a defect rather than numerical drift.


In [ ]:
with torch.no_grad():
    reference = hf(input_ids=ids).logits
adapter_logits = seam.forward(engine, ids)
print(f"max|adapter - huggingface| = {(adapter_logits - reference).abs().max().item():.3e}")


## 3. Intervene, at each declared position scope

The backend declares `LAST_TOKEN` and `ALL_POSITIONS`. The scope is not a detail: interp-engine
steers every prompt position by default while interpretune's default is the last token, so passing the
engine's default through would apply your vector at five positions when you asked for one — and the
forward would complete and look plausible. The seam expresses the scope as an explicit position mask.


In [ ]:
from interpretune.analysis.backends.interventions import PositionScope

torch.manual_seed(0)
vector = torch.randn(hf.config.n_embd) * 0.5
spec = seam.steer_spec(vector=vector, layer=5, coeff=1.0, method="additive", point="resid_post")
baseline = seam.forward(engine, ids)

for scope in (PositionScope.LAST_TOKEN, PositionScope.ALL_POSITIONS):
    steered = seam.steer(engine, ids, [(scope, spec)])
    moved = ((steered - baseline).abs().amax(dim=-1) > 1e-6)[0]
    print(f"{scope.value:14} moved positions: {moved.nonzero().flatten().tolist()} of {ids.shape[-1]}")


## 4. What it refuses, and why that is the point

TransformerLens fires `blocks.{i}.hook_mlp_in` on the residual stream **before** the block norm, while
the sublayer's actual argument is the norm's **output**. On gemma-3-1b-it layer 5 those tensors share a
cosine similarity of 0.088, and published SAE dashboards have been generated from the wrong one.

So the adapter refuses that name rather than resolving it to the tensor you probably meant, and names
what to ask for instead. A refusal you can read beats a plausible number you cannot check.


In [ ]:
UnmappableHookError = loaded_adapter_module(REPO).UnmappableHookError

try:
    seam.point_for("blocks.5.hook_mlp_in", engine)
except UnmappableHookError as refusal:
    print("refused, as intended:\n")
    print(refusal)

# The name it points you to resolves cleanly, and is a different tensor:
print("\nblocks.5.mlp.hook_in ->", seam.point_for("blocks.5.mlp.hook_in", engine))


## Where to look next

- The component's own documentation states what it declares and what it refuses, which this card
  cannot tell you: the publisher never executes the entrypoint, so runtime capabilities are not
  derivable from the manifest.
- `loaded.skipped` above is the honest answer to "why do I have fewer compositions than the card
  advertised". It is reported rather than silent, by design.
